In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.model_selection import RandomizedSearchCV
import joblib

In [2]:
import os
import sys

# Absolute path to project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Change working directory to project root
os.chdir(PROJECT_ROOT)

# Add project root to Python path
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\venut\OneDrive\Documents\Projects\credit_risk_project


In [3]:
cols = pd.read_csv("data/processed/dataset.csv", nrows=0).columns.tolist()

In [4]:
required_cols = [
    "loan_status", "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose", "dti",
    "delinq_2yrs", "fico_range_low", "fico_range_high",
    "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "application_type"
]

In [5]:
chunks = []
chunk_size = 100_000

for chunk in pd.read_csv(
    "data/processed/dataset.csv",
    usecols=required_cols,
    chunksize=chunk_size,
    low_memory=False
):
    # Keep only clear outcomes
    chunk = chunk[chunk["loan_status"].isin(
        ["Fully Paid", "Charged Off", "Default"]
    )]
    
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
df.shape


(1345350, 22)

In [6]:
df["target"] = df["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1
})

In [7]:
X = df.drop(columns=["target", "loan_status"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [8]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

In [9]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


In [10]:
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [11]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       0.81      0.99      0.89    215350
           1       0.54      0.05      0.09     53720

    accuracy                           0.80    269070
   macro avg       0.68      0.52      0.49    269070
weighted avg       0.75      0.80      0.73    269070

ROC-AUC: 0.7090534861515678


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

results = {}

for name, clf in models.items():
    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", clf)
    ])
    
    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1
    )
    
    results[name] = scores.mean()
    print(f"{name} ROC-AUC: {scores.mean():.4f}")


Logistic Regression ROC-AUC: 0.7094
Decision Tree ROC-AUC: 0.6996
Random Forest ROC-AUC: 0.7056


In [13]:
best_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [14]:
param_dist = {
    "classifier__n_estimators": [100, 150],
    "classifier__max_depth": [6, 8],
    "classifier__min_samples_split": [2, 5]
}

random_search = RandomizedSearchCV(
    estimator=best_model,
    param_distributions=param_dist,
    n_iter=4,               # VERY IMPORTANT (keep small)
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=1,               # CRITICAL FIX
    verbose=2
)

random_search.fit(X_train, y_train)

print("Best Params:", random_search.best_params_)
print("Best CV ROC-AUC:", random_search.best_score_)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END classifier__max_depth=6, classifier__min_samples_split=2, classifier__n_estimators=150; total time=  54.5s
[CV] END classifier__max_depth=6, classifier__min_samples_split=2, classifier__n_estimators=150; total time=  54.4s
[CV] END classifier__max_depth=6, classifier__min_samples_split=2, classifier__n_estimators=150; total time=  54.6s
[CV] END classifier__max_depth=8, classifier__min_samples_split=2, classifier__n_estimators=150; total time= 1.4min
[CV] END classifier__max_depth=8, classifier__min_samples_split=2, classifier__n_estimators=150; total time= 1.4min
[CV] END classifier__max_depth=8, classifier__min_samples_split=2, classifier__n_estimators=150; total time= 1.4min
[CV] END classifier__max_depth=6, classifier__min_samples_split=2, classifier__n_estimators=100; total time=  39.2s
[CV] END classifier__max_depth=6, classifier__min_samples_split=2, classifier__n_estimators=100; total time= 1.1min
[CV] END cla

In [15]:
final_model = random_search.best_estimator_

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       0.89      0.60      0.71    215350
           1       0.30      0.70      0.42     53720

    accuracy                           0.62    269070
   macro avg       0.59      0.65      0.57    269070
weighted avg       0.77      0.62      0.66    269070

Test ROC-AUC: 0.7048568700003682


In [16]:
joblib.dump(final_model, "credit_risk_model.pkl")

['credit_risk_model.pkl']

In [19]:
# Remove target columns
df = df.drop(columns=["loan_status", "target"], errors="ignore")

df.head()


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,dti,delinq_2yrs,fico_range_low,fico_range_high,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type
0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,...,5.91,0.0,675.0,679.0,7.0,0.0,2765.0,29.7,13.0,Individual
1,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,...,16.06,1.0,715.0,719.0,22.0,0.0,21470.0,19.2,38.0,Individual
2,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,...,10.78,0.0,695.0,699.0,6.0,0.0,7869.0,56.2,18.0,Joint App
3,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,...,25.37,1.0,695.0,699.0,12.0,0.0,21929.0,64.5,35.0,Individual
4,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,...,10.20,0.0,690.0,694.0,5.0,0.0,8822.0,68.4,6.0,Individual
